# Agent Flow Debug (Step-by-Step)

This notebook runs the agents **one-by-one** (no FastAPI, no LangGraph) so you can see each agent's inputs/outputs.

Prereqs:
- Put keys in `.env.local` (recommended): `OPENAI_API_KEY` and `TAVILY_API_KEY` (and/or `ANTHROPIC_API_KEY` depending on `config/agents.yaml`).
- Install deps: `pip install -r requirements.txt`

Notes:
- This is intentionally minimal: no looping, no retries, no special error handling.
- Each agent reads provider/model from `config/agents.yaml`.


In [ ]:
import sys
from pathlib import Path

# Ensure repo root is on sys.path so `import src.*` works reliably.
repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.env import load_env
load_env()


In [ ]:
import json
from datetime import datetime

from src.state.schema import PipelineState
from src.agents.coordinator import CoordinatorAgent
from src.agents.research import ResearchAgent
from src.agents.analysis import AnalysisAgent
from src.agents.writing import WritingAgent
from src.agents.quality import QualityAgent


def state_summary(state: dict) -> dict:
    def _n(key: str) -> int:
        v = state.get(key)
        return len(v) if isinstance(v, list) else 0

    return {
        "run_id": state.get("run_id"),
        "topic": state.get("topic"),
        "current_agent": state.get("current_agent"),
        "pipeline_status": state.get("pipeline_status"),
        "quality_iteration": state.get("quality_iteration"),
        "quality_verdict": state.get("quality_verdict"),
        "quality_score": state.get("quality_score"),
        "counts": {
            "research_queries": _n("research_queries"),
            "raw_sources": _n("raw_sources"),
            "deduplicated_sources": _n("deduplicated_sources"),
            "clinical_claims": _n("clinical_claims"),
            "evidence_gaps": _n("evidence_gaps"),
            "citations": _n("citations"),
            "quality_issues": _n("quality_issues"),
        },
        "report_word_count": state.get("report_word_count"),
        "error_message": state.get("error_message"),
    }


def build_initial_state(
    *,
    topic: str,
    scope_instructions: str = "",
    target_audience: str = "clinical practitioners",
    report_format: str = "clinical_brief",
    max_quality_iterations: int = 3,
) -> PipelineState:
    return PipelineState(
        run_id="notebook-" + datetime.now().strftime("%Y%m%d-%H%M%S"),
        topic=topic,
        scope_instructions=scope_instructions,
        target_audience=target_audience,
        report_format=report_format,
        requested_at=datetime.now(),
        research_queries=[],
        scope_boundaries={"in_scope": [], "out_of_scope": []},
        priority_subtopics=[],
        raw_sources=[],
        deduplicated_sources=[],
        research_summary="",
        clinical_claims=[],
        evidence_gaps=[],
        contradictions=[],
        statistical_findings=[],
        analysis_narrative="",
        report_sections={},
        report_markdown="",
        citations=[],
        report_word_count=0,
        quality_issues=[],
        quality_verdict="pending",
        quality_score=0.0,
        revision_instructions="",
        quality_iteration=0,
        max_quality_iterations=max_quality_iterations,
        should_revise=False,
        current_agent="",
        agent_history=[],
        pipeline_status="pending",
        error_message=None,
    )


In [ ]:
# Edit these, then re-run cells top-to-bottom.
TOPIC = "Effectiveness of telemedicine interventions for Type 2 diabetes management in adults"
SCOPE = "Focus on adults 18+, exclude pediatric cases. Prefer evidence from 2018-2026."
AUDIENCE = "clinical practitioners"
REPORT_FORMAT = "clinical_brief"  # or "full_report"
MAX_QUALITY_ITERATIONS = 3

state = build_initial_state(
    topic=TOPIC,
    scope_instructions=SCOPE,
    target_audience=AUDIENCE,
    report_format=REPORT_FORMAT,
    max_quality_iterations=MAX_QUALITY_ITERATIONS,
)

state_summary(state)


## 1) Coordinator Agent
Generates `research_queries`, `scope_boundaries`, and `priority_subtopics`.

In [ ]:
coordinator = CoordinatorAgent()
coordinator._current_run_id = state.get("run_id")

state = await coordinator.execute(state)

print("Coordinator summary:")
print(json.dumps({
    "research_queries": state.get("research_queries"),
    "scope_boundaries": state.get("scope_boundaries"),
    "priority_subtopics": state.get("priority_subtopics"),
}, indent=2))
print("\nState:")
print(json.dumps(state_summary(state), indent=2))


## 2) Research Agent
Runs Tavily searches for `research_queries`, deduplicates sources, and writes `research_summary`.

Requires `TAVILY_API_KEY` in environment.

In [ ]:
research = ResearchAgent()
research._current_run_id = state.get("run_id")

state = await research.execute(state)

print("Research summary:")
print(state.get("research_summary"))

sources = state.get("deduplicated_sources", [])
print(f"\nDeduplicated sources: {len(sources)}")
for i, s in enumerate(sources[:5], 1):
    print(f"{i}. [{s.get('evidence_level')}] {s.get('title')} ({s.get('relevance_score')})")
    print(f"   {s.get('url')}")

print("\nState:")
print(json.dumps(state_summary(state), indent=2))


## 3) Analysis Agent
Extracts `clinical_claims`, `evidence_gaps`, `contradictions`, `statistical_findings`, and `analysis_narrative`.

In [ ]:
analysis = AnalysisAgent()
analysis._current_run_id = state.get("run_id")

state = await analysis.execute(state)

print("Analysis narrative:\n")
print(state.get("analysis_narrative"))

claims = state.get("clinical_claims", [])
print(f"\nClinical claims: {len(claims)}")
for i, c in enumerate(claims[:5], 1):
    print(f"{i}. {c.get('claim')} ({c.get('evidence_level')})")
    if c.get("effect_size"):
        print(f"   effect_size: {c.get('effect_size')}")
    if c.get("p_value"):
        print(f"   p_value: {c.get('p_value')}")
    if c.get("source_urls"):
        print(f"   sources: {c.get('source_urls')}")

gaps = state.get("evidence_gaps", [])
print(f"\nEvidence gaps: {len(gaps)}")
for g in gaps[:10]:
    print(f"- {g}")

print("\nState:")
print(json.dumps(state_summary(state), indent=2))


## 4) Writing Agent
Produces `report_sections`, `report_markdown`, `citations`, and `report_word_count`.

In [ ]:
writing = WritingAgent()
writing._current_run_id = state.get("run_id")

state = await writing.execute(state)

print(f"Word count: {state.get('report_word_count')}")
print(f"Citations: {len(state.get('citations', []) or [])}")

print("\nReport preview:\n")
report = state.get("report_markdown", "") or ""
print(report[:2000])

print("\nState:")
print(json.dumps(state_summary(state), indent=2))


## 5) Quality Agent
Scores the report and returns a verdict and issues.

This notebook runs quality once (no revision loop).

In [ ]:
quality = QualityAgent()
quality._current_run_id = state.get("run_id")

state = await quality.execute(state)

print(f"Quality verdict: {state.get('quality_verdict')}")
print(f"Quality score: {state.get('quality_score')}")

issues = state.get("quality_issues", [])
print(f"\nIssues: {len(issues)}")
for i, issue in enumerate(issues[:20], 1):
    print(f"{i}. [{issue.get('severity')}] {issue.get('section')}: {issue.get('description')}")
    rec = issue.get('recommendation')
    if rec:
        print(f"   recommendation: {rec}")

print("\nRevision instructions:")
print(state.get("revision_instructions"))

print("\nState:")
print(json.dumps(state_summary(state), indent=2))
